# Plant Detection using TensorFlow Object Detection API

This notebook demonstrates training and evaluation of an object detection model
for plant instance detection on the :contentReference[oaicite:0]{index=0} dataset.

We fine-tune a pre-trained SSD model:
- Model: SSD MobileNet V2 (320×320)
- Framework: TensorFlow Object Detection API (TF 2.11)
- Task: Crop vs Weed detection

The notebook covers:
1. Environment setup
2. Dataset loading
3. Model configuration
4. Training and evaluation
5. Qualitative inference results

## Environment Setup

This experiment is designed for:
- Python 3.10
- TensorFlow 2.11
- CUDA-enabled GPU

On Kaggle:
Enable "Pin to original environment" to avoid version drift.

In [ ]:
!python -V

In [ ]:
!pip install --no-cache-dir --no-deps \
  tf_slim \
  pycocotools \
  lvis \
  contextlib2 \
  gin-config \
  tf-models-official==2.13.2 \
  git+https://github.com/frdiener/agri-vision-edge.git

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
from dataclasses import asdict
import tensorflow as tf

from agri_vision_edge.third_party import setup_tensorflow_models
setup_tensorflow_models()

from agri_vision_edge.experiment import (
    ExperimentManifest,
    capture_environment,
)
from agri_vision_edge.experiment import (
    AugmentationConfig,
    FineTuneConfig,
)
from agri_vision_edge.tfod.tfod_metrics import load_tfod_best_metrics
from agri_vision_edge.evaluation.tensorboard import load_event_scalars
from agri_vision_edge.evaluation.curves import (
    available_tags,
    plot_metric_curves,
)
from agri_vision_edge.tfod import export_all

from object_detection import model_lib_v2

In [ ]:
EXPERIMENT_ROOT = Path(
    "/kaggle/working"
)

FINETUNE_PATH = (
    EXPERIMENT_ROOT / "finetune"
)

FINETUNE_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

EXPORTS_PATH = (
    EXPERIMENT_ROOT / "exports"
)

MANIFEST_PATH = (
    EXPERIMENT_ROOT / "manifest.json"
)

In [ ]:
manifest = ExperimentManifest(
    name="ssd-mobilenet-v2_sc_phenobench-tiled_320x320",
    task="object_detection",
)

manifest.set_environment(
    platform="kaggle",
    **capture_environment(),
)

In [ ]:
config = FineTuneConfig(
    batch_size=16,
    learning_rate_base=0.001,
    warmup_learning_rate=0.0003,

    num_steps=30_000,

    warmup_steps=1000,

    early_stopping_patience=60,
    early_stopping_min_delta=0.0,

    image_size=320,

    #
    # Anchor tuning
    #

    anchor_min_scale=0.02,
    anchor_max_scale=0.28,

    anchor_aspect_ratios=(
        0.33,
        0.5,
        1.0,
        2.0,
        3.0,
    ),

    #
    # Matcher thresholds
    #

    matched_threshold=0.4,
    unmatched_threshold=0.3,

    #
    # Data augmentation
    #

    augmentation=AugmentationConfig(
        #
        # Crop
        #

        random_crop=True,

        crop_min_object_covered=1.0,
        crop_min_area=0.6,
        crop_max_area=1.0,
        crop_overlap_thresh=0.3,

        #
        # Geometric invariance
        #

        horizontal_flip=True,
        horizontal_flip_probability=0.5,

        vertical_flip=True,
        vertical_flip_probability=0.5,

        rotation90=True,
        rotation90_probability=0.5,

        #
        # Scale / zoom invariance
        #

        zoom_range=(
            0.8,
            1.2,
        ),

        #
        # Photometric augmentation
        #

        brightness_max_delta=0.15,

        contrast_range=(
            0.8,
            1.2,
        ),

        saturation_range=(
            0.8,
            1.2,
        ),

        hue_max_delta=0.2,
        
        #
        # Compression robustness
        #

        # jpeg_quality_range=(50,100),
    ),

    #
    # NMS
    #

    nms_score_threshold=0.05,

    nms_iou_threshold=0.5,

    max_detections_per_class=100,
    max_total_detections=100,
)

manifest.add_stage(
    "finetune",
    config=asdict(config),
)

'defined'

## Dataset

We use preprocessed TFRecord files derived from :contentReference[oaicite:1]{index=1}.

Additionally, raw images are used for qualitative evaluation.

In [ ]:
manifest.set_dataset(
    name="phenobench",

    train_split="train",

    validation_split="val",

    num_classes=1,
)

dataset_dir = Path("/kaggle/input/datasets/freimutdiener/sc-phenobench-tiled")
dataset_raw_dir = Path("/kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench")

label_map_path = dataset_dir / "label_map.pbtxt"
train_record = dataset_dir / "train.record"
val_record = dataset_dir / "val.record"

test_imgs = list((dataset_raw_dir / "test" / "images").glob("*.png"))

print(f"{len(test_imgs)} test images loaded")

## Model Configuration

We fine-tune a pre-trained SSD model:
- Backbone: MobileNetV2
- Feature extractor: FPNLite
- Input size: 320×320

We adapt:
- number of classes
- learning rate schedule
- dataset paths

In [ ]:
model_name = "ssd_mobilenet_v2_320x320_coco17_tpu-8"

manifest.set_checkpoint(
    model_name=model_name,
    pretrained_dataset="coco17",

    source="tensorflow_model_zoo",
)

MODEL_DIR = (Path("/kaggle/input/models/freimutdiener/"
                  "ssd-mobilenet-v2-320x320/tensorflow2/coco17/1")
             / model_name)

In [ ]:
PIPELINE_CONFIG = FINETUNE_PATH / "pipeline.config"

from agri_vision_edge.tfod import configure_ssd_pipeline

configure_ssd_pipeline(
    config=config,
    config_path=MODEL_DIR / "pipeline.config",
    output_path=PIPELINE_CONFIG,
    train_record=train_record,
    val_record=val_record,
    label_map=label_map_path,
    checkpoint_path=MODEL_DIR / "checkpoint" / "ckpt-0",
    num_classes=1,
)

manifest.add_artifact(
    "finetune/pipeline.config",

    artifact_type="pipeline_config",

    stage="finetune",
)

## Training

We train for 20,000 steps using fine-tuning from COCO weights.

In [ ]:
model_lib_v2.train_loop(
  pipeline_config_path=PIPELINE_CONFIG,
  model_dir=str(FINETUNE_PATH),
  checkpoint_every_n=100,
  checkpoint_max_to_keep=1,
  early_stopping_patience=config.early_stopping_patience,
  early_stopping_min_delta=config.early_stopping_min_delta,
)

In [ ]:
BEST_METRICS_JSON = FINETUNE_PATH / "best_metric.json"

metrics = load_tfod_best_metrics(
    BEST_METRICS_JSON
)

manifest.update_stage(
    "finetune",

    metrics={
        "best_coco_metrics":
            metrics.to_dict()
    }
)

manifest.update_stage(
    "finetune",

    artifacts={
        "best_metrics_json":
            "finetune/best_metric.json"
    }
)

## Training Metrics and Learning Curves

The TensorFlow Object Detection API writes training metrics
as TensorBoard event files during optimization.

We parse these logs to generate publication-quality plots
for:

- total training loss
- classification loss
- localization loss
- learning rate schedule
- training throughput

These figures can later be exported directly for inclusion
in reports, presentations, or academic theses.

In [ ]:
GRAPHS_PATH = FINETUNE_PATH / "graphs"
GRAPHS_PATH.mkdir(
    parents=True,
    exist_ok=True,
)

train_df = load_event_scalars(
    FINETUNE_PATH / "train"
)

eval_df = load_event_scalars(
    FINETUNE_PATH / "eval_on_train"
)

print("Available TensorBoard tags:")
for tag in available_tags(train_df):
    print("-", tag)

In [ ]:
tags = [
    "Loss/total_loss",
    "Loss/classification_loss",
    "Loss/localization_loss",
    "Loss/regularization_loss",
]

fig, ax = plot_metric_curves(
    df=train_df,
    tags=tags,
    title="Training Loss",
    ylabel="Loss",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "training_loss_curves.pdf",
)

display(fig)

In [ ]:

fig, ax = plot_metric_curves(
    df=train_df,
    tags=["learning_rate"],
    title="Learning Rate Schedule",
    ylabel="Learning Rate",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "learning_rate_schedule.pdf",
)

display(fig)

In [ ]:
fig, ax = plot_metric_curves(
    df=train_df,
    tags=["steps_per_sec"],
    title="Training Throughput",
    ylabel="Steps / Second",
    smoothing=0.5,
    save_path=GRAPHS_PATH / "training_throughput.pdf",
)

display(fig)

The generated figures are also exported as PDF files:

- `training_loss_curves.pdf`
- `learning_rate_schedule.pdf`
- `training_throughput.pdf`

These vector graphics are suitable for direct inclusion
in scientific publications and LaTeX-based theses.

## Validation Metrics

We evaluate all saved checkpoints using standard
COCO-style object detection metrics.

This enables analysis of:

- convergence behavior
- validation performance evolution
- checkpoint selection
- potential overfitting

In [ ]:
print("Available TensorBoard tags:")
for tag in available_tags(eval_df):
    print("-", tag)

In [ ]:
tags = [
    "Loss/total_loss",
    "Loss/classification_loss",
    "Loss/localization_loss",
    "Loss/regularization_loss",
]

fig, ax = plot_metric_curves(
    df=eval_df,
    tags=tags,
    title="Evaluation Loss",
    ylabel="Loss",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_loss_curves.pdf",
)

display(fig)

In [ ]:
tags = [
    "DetectionBoxes_Precision/mAP",
    "DetectionBoxes_Precision/mAP@.50IOU",
    "DetectionBoxes_Precision/mAP@.75IOU",
    "DetectionBoxes_Recall/AR@100",
]

fig, ax = plot_metric_curves(
    df=eval_df,
    tags=tags,
    title="Evaluation Precision",
    ylabel="Score",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_precision_curves.pdf",
)

display(fig)

In [ ]:
tags = [
    "DetectionBoxes_Recall/AR@1",
    "DetectionBoxes_Recall/AR@10",
    "DetectionBoxes_Recall/AR@100",
]

fig, ax = plot_metric_curves(
    df=eval_df,
    tags=tags,
    title="Evaluation Recall",
    ylabel="Score",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_recall_curves.pdf",
)

display(fig)

In [ ]:
tags = [
    "DetectionBoxes_Precision/mAP (small)",
    "DetectionBoxes_Precision/mAP (medium)",
    "DetectionBoxes_Precision/mAP (large)",
]

fig, ax = plot_metric_curves(
    df=eval_df,
    tags=tags,
    title="Evaluation Precision per Size",
    ylabel="Score",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_precision_size_curves.pdf",
)

display(fig)

In [ ]:
tags = [
    "DetectionBoxes_Recall/AR@100 (small)",
    "DetectionBoxes_Recall/AR@100 (medium)",
    "DetectionBoxes_Recall/AR@100 (large)",
]

fig, ax = plot_metric_curves(
    df=eval_df,
    tags=tags,
    title="Evaluation Recall per Size",
    ylabel="Score",
    smoothing=0.6,
    save_path=GRAPHS_PATH / "eval_recall_size_curves.pdf",
)

display(fig)

In [ ]:
manifest.add_artifact(
    "finetune/graphs",

    artifact_type="evaluation_plots",

    stage="finetune",
)

## Model Export

We export the trained model in two formats:

- a standard TensorFlow SavedModel for generic TensorFlow
  inference workflows
- a TensorFlow Lite–compatible export graph for quantization
  and embedded deployment

The TensorFlow Lite export path uses TensorFlow Object
Detection's dedicated TFLite exporter, which rewrites the
graph for improved compatibility with:

- TensorFlow Lite conversion
- post-training quantization
- embedded accelerators
- NPU delegates such as TIM-VX / Teflon

This export path typically reduces dynamic TensorFlow ops
and improves deployment compatibility on edge devices.

In [ ]:
export_all(
    pipeline_config_path=str(PIPELINE_CONFIG),

    checkpoint_dir=str(FINETUNE_PATH),

    export_dir=str(EXPORTS_PATH),

    config_override="",
    input_type="image_tensor",

    max_detections=config.max_total_detections,
)

## Register artifacts and save Manifest

In [ ]:
manifest.add_artifact(
    "finetune/train",
    artifact_type="tensorboard_train",
    stage="finetune",
)

manifest.add_artifact(
    "finetune/eval_on_train",
    artifact_type="tensorboard_eval",
    stage="finetune",
)

manifest.add_artifact(
    "exports/exported_model",
    artifact_type="tfod_exported_model",
    stage="finetune",
)

manifest.add_artifact(
    "exports/tflite_graph",
    artifact_type="tflite_graph",
    stage="finetune",
)

manifest.save(MANIFEST_PATH)

## Qualitative Evaluation

We visualize predictions on unseen test images.

In [ ]:
%matplotlib inline

from agri_vision_edge.tfod.inference import (
    load_saved_model,
    load_label_map,
    detect_image,
)

detect_fn = load_saved_model(
    "/kaggle/working/exports/exported_model/saved_model"
)

category_index = load_label_map(label_map_path)

for image_path in test_imgs[:10]:

    vis, _ = detect_image(
        detect_fn=detect_fn,
        image_path=image_path,
        category_index=category_index,
        image_size=320,
        score_threshold=0.5,
        max_boxes=60,
    )
#     display(vis)

    plt.figure(figsize=(16, 16))

    plt.imshow(vis)

    plt.axis("off")

    plt.show()

    plt.close(fig)

## Discussion

The model demonstrates:

- Successful localization of plant instances
- Overlapping detections reduced via NMS
- Sensitivity to small objects (PhenoBench-specific challenge)

Limitations:
- Performance depends strongly on resolution (320×320)
- Dense scenes produce multiple candidate detections
- Further improvements possible via:
  - anchor tuning
  - longer training
  - quantization-aware training (QAT)

Future work:
- INT8 deployment via TFLite
- Real-time inference optimization